In [1]:
!nvidia-smi

Tue Apr 28 00:09:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P0             81W /  400W |   33200MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
!pip install -U vllm

### Qwen/Qwen3.6-27B-FP8

In [6]:
!ps aux | grep vllm
!nvidia-smi

root       12466 52.7  2.4 14376732 2124108 ?    Sl   00:10   0:32 python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3.6-27B-FP8 --served-model-name Qwen/Qwen3.6-27B-FP8 --port 8081 --max-model-len 4096 --gpu-memory-utilization 0.75 --trust-remote-code
root       12881  0.0  0.0   7376  3564 ?        S    00:11   0:00 /bin/bash -c ps aux | grep vllm
root       12883  0.0  0.0   6484  2364 ?        S    00:11   0:00 grep vllm
Tue Apr 28 00:11:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                       

In [7]:
!pkill -f vllm
!nvidia-smi

Tue Apr 28 00:11:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P0             61W /  400W |   29804MiB /  40960MiB |      3%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [8]:
!nohup python -m vllm.entrypoints.openai.api_server \
  --model Qwen/Qwen3.6-27B-FP8 \
  --served-model-name Qwen/Qwen3.6-27B-FP8 \
  --port 8081 \
  --max-model-len 4096 \
  --gpu-memory-utilization 0.75 \
  --trust-remote-code \
  > vllm_8081.log 2>&1 &

In [9]:
!tail -n 50 vllm.log

(APIServer pid=8081)                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(APIServer pid=8081)   File "/usr/lib/python3.12/contextlib.py", line 210, in __aenter__
(APIServer pid=8081)     return await anext(self.gen)
(APIServer pid=8081)            ^^^^^^^^^^^^^^^^^^^^^
(APIServer pid=8081)   File "/usr/local/lib/python3.12/dist-packages/vllm/entrypoints/openai/api_server.py", line 136, in build_async_engine_client_from_engine_args
(APIServer pid=8081)     async_llm = AsyncLLM.from_vllm_config(
(APIServer pid=8081)                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
(APIServer pid=8081)   File "/usr/local/lib/python3.12/dist-packages/vllm/v1/engine/async_llm.py", line 217, in from_vllm_config
(APIServer pid=8081)     return cls(
(APIServer pid=8081)            ^^^^
(APIServer pid=8081)   File "/usr/local/lib/python3.12/dist-packages/vllm/v1/engine/async_llm.py", line 146, in __init__
(APIServer pid=8081)     self.engine_core = EngineCoreClient.make_async_mp_client(
(APIServer pid=8081)   

In [11]:
import requests

url = "http://localhost:8081/v1/chat/completions"

payload = {
    "model": "Qwen/Qwen3.6-27B-FP8",
    "messages": [
        {"role": "user", "content": "Explain what vLLM is in simple terms."}
    ],
    "temperature": 0.7,
    "max_tokens": 200
}

response = requests.post(url, json=payload, timeout=120)

print(response.json()["choices"][0]["message"]["content"])

ConnectionError: HTTPConnectionPool(host='localhost', port=8081): Max retries exceeded with url: /v1/chat/completions (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7bec14262c30>: Failed to establish a new connection: [Errno 111] Connection refused'))

###